# v0 — `fb_posts` → structured listings (read-only, 10 rows)

This notebook is a **visual checkpoint** before the full v1 Supabase pipeline. It:

1. reads **10 real `fb_posts` rows** from Supabase (READ-ONLY, no writes anywhere),
2. runs the shared `extract.py` engine in **ONE batched LLM call**,
3. shows the transformation step by step: raw text → raw model JSON → normalized
   40-field table → key-fields side-by-side → summary.

It calls the *exact same engine* the v1 pipeline uses, so what you see here is what v1
will write to `listings_parsed` at scale. **Nothing is written to the database.**


## Setup
Locate the project root (so `import extract` works), load `.env`, and check keys.

In [ ]:
import sys, json
from pathlib import Path

# Find the project root (this notebook lives in notebooks/; extract.py is at the root).
cwd = Path.cwd()
ROOT = cwd if (cwd / "extract.py").exists() else cwd.parent
assert (ROOT / "extract.py").exists(), f"could not find extract.py from {cwd}"
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import os
import pandas as pd
import extract

missing = [k for k in ("ANTHROPIC_API_KEY", "SUPABASE_URL", "SUPABASE_ANON_KEY")
           if not os.environ.get(k)]
assert not missing, f"Missing in .env: {missing}. Add them and re-run."
print("Project root:", ROOT)
print("extract.py:", len(extract.MODEL_FIELDS), "fields, parser", extract.PARSER_VERSION)
print("env OK")


## Step 1 — read 10 raw `fb_posts` rows (READ-ONLY)
A single `.limit(10)` query. No writes.

In [ ]:
from supabase import create_client

client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_ANON_KEY"])
res = client.table("fb_posts").select("id,source,text,created_at").limit(10).execute()
raw_rows = res.data or []
print(f"fetched {len(raw_rows)} rows")

raw_df = pd.DataFrame(raw_rows)
# show the text in full-ish for inspection
pd.set_option("display.max_colwidth", 200)
raw_df[["id", "source", "created_at", "text"]]


## Step 2 — run the extraction (ONE LLM call)

`call_claude_batch(texts)` sends all 10 posts in a single request and returns a JSON
array (one object per post). This is the happy path of `extract.extract_batch()`, which
in production also (a) skips texts shorter than 15 chars with **no** call and (b) falls
back to per-row calls if the batch fails — neither should trigger on 10 normal rows.


In [ ]:
texts = [r.get("text") or "" for r in raw_rows]

raw_objects = extract.call_claude_batch(texts)   # <-- the single LLM call
print(f"model returned {len(raw_objects)} objects for {len(texts)} posts")

if len(raw_objects) != len(texts):
    print("! length mismatch -> using robust extract_batch() (adds per-row fallback)")
    normalized = extract.extract_batch(texts)
    raw_objects = None
else:
    normalized = [extract._coerce(o) for o in raw_objects]
print("normalized rows:", len(normalized))


## Step 3 — what the model returned (raw JSON, first 2 posts)
Before any normalization/coercion.

In [ ]:
if raw_objects is not None:
    print(json.dumps(raw_objects[:2], indent=2, ensure_ascii=False))
else:
    print("(batch fell back to per-row; raw array not available)")


## Step 4 — normalized 40-field table

After `_coerce`: enums lowercased (invalid → fallback), list fields joined, `null` kept
as unknown. These columns are exactly what v1 will upsert into `listings_parsed`.


In [ ]:
parsed_df = pd.DataFrame(normalized)[extract.MODEL_FIELDS]
parsed_df


## Step 5 — side-by-side: raw text → key extracted fields
The quick human sanity-check.

In [ ]:
key_cols = ["is_offer", "discard_reason", "price_thb", "bedrooms",
            "property_type", "area_canonical", "post_language", "parse_confidence"]
side = parsed_df[key_cols].copy()
side.insert(0, "text", [ (r.get("text") or "")[:120] for r in raw_rows ])
side.insert(0, "id", [r.get("id") for r in raw_rows])
pd.set_option("display.max_colwidth", 130)
side


## Step 6 — summary

How the 10 rows classify, and how many survive the offers filter
(`discard_reason IS NULL`) that v1 will use for your actual house search.


In [ ]:
print("discard_reason:")
print(parsed_df["discard_reason"].fillna("(kept / offer)").value_counts(), "\n")
print("area_canonical:")
print(parsed_df["area_canonical"].value_counts(), "\n")
print("property_type:")
print(parsed_df["property_type"].value_counts(), "\n")

kept = parsed_df["discard_reason"].isna().sum()
print(f"kept as offers (discard_reason is null): {kept} / {len(parsed_df)}")


## Verdict

If the key fields above look right (prices/beds/areas sensible on real offers; obvious
non-listings / wanted / for-sale rows flagged with the right `discard_reason`), the
engine is good and we proceed to **v1** — the full incremental pipeline that writes
`listings_parsed` in Supabase. If something looks off, we tune `extract.py`'s prompt and
re-run this notebook (still just 1 LLM call) before touching the database.
